# Smile Intro


This document demonstrates how to work with Arduino and how it can be controlled from Python.


The Arduino SMAIL firmware includes an early version of the Cmd command interpreter and the Ticker library, which executes tasks at predefined time intervals. A module is also provided that allows reading and writing several float variables via Cmd; these variables control a pseudo-data generator.


Communication between the PC and the Arduino is performed through a serial channel, in this case via USB. On the computer side, a serial terminal program (for example, the Arduino Serial Monitor or PlatformIO Monitor) can be used to send commands to the Arduino and receive its responses.


In a Python environment, the PykDev module can be used to send commands and parse their responses. An extended version of Cmd is also provided, including a built-in hlp command that lists all available commands. In this case, PykiDev can be used as an extension of Pykiba, mapping Arduino commands to Python functions.

## Setup connections with a computer

In general, the drivers can be used in two ways. One option is to import them directly into the user software, like any other module. The other option is to run them as a ZeroRPC server, to which the client software connects.

In both cases, the result is an object that is used in exactly the same way.



### Import module(driver) in the user program 





In [6]:

from pykidev import Pykiba, PykiDev, search_by_manufacturer,  serial_ports_list
import time
from pprint import pprint


name = "Arduino (www.arduino.cc)"       # Linux driver name for the Arduino Mega2560

ports_found = serial_ports_list()      # Return a list of all available serial ports

arduino_port = search_by_manufacturer(ports_found, name)                                      

print(f'\n\n  Computer comport of the arduino is: {arduino_port}')  

# Initialize communication with PykiDev..

ard = PykiDev(port = arduino_port, baudrate = 9600) # by default baudrate = 9600 , colud be 115200 

#time.sleep(2) #must be a delay 

print(f'\n\n Returns the name of the project: {ard.command("hello")}') # This provides access to all Arduino commands
print(f'\n   The name of the project: {ard.hello() = }') # The commands are mapped to Python functions 

# In mode 1, the LED blinks twice
print(f'\n   Sets mode of operation of the Arduino. Returns setted mode:  { ard.command("mode",1) =}') 

# The 'mode' command can be called like a standard Python function: mode()
print(f'\n   Sets mode of operation of the Arduino. Returns setted mode:  { ard.mode(2) = } ' ) 

print('\n\n Global variables:')
print('[name, value, min, max, update flag]')
pprint(ard.varlst())#

#del ard  # To close comport and free it for next example



0.   None   /dev/ttyS0
1.   Arduino (www.arduino.cc)   /dev/ttyACM0


  Computer comport of the arduino is: /dev/ttyACM0


 Returns the name of the project: ['Arduino', 'SMILE']

   The name of the project: ard.hello() = ['Arduino', 'SMILE']

   Sets mode of operation of the Arduino. Returns setted mode:   ard.command("mode",1) =1

   Sets mode of operation of the Arduino. Returns setted mode:   ard.mode(2) = 2 


 Global variables:
[name, value, min, max, update flag]
[['t', 524.0, 0.0, 0.0, 1],
 ['f2', 3.54, 0.0, 0.0, 1],
 ['f1', 3.54, 0.0, 0.0, 1],
 ['a2', 3.1, 0.0, 10.0, 1],
 ['a1', 3.1, 0.0, 10.0, 1],
 ['w2', 0.1, 0.01, 10000.0, 1],
 ['w1', 0.1, 0.01, 10000.0, 1]]


In [7]:
print(f'\n\n All available Arduino comnds: \n\n {ard.hlp() = }')



 All available Arduino comnds: 

 ard.hlp() = ['varlst', 't', 'f2', 'f1', 'a2', 'a1', 'w2', 'w1', 'mode', 'hello', 'echo', 'hlp']


In [8]:
del ard  # To close comport and free it for next example

### PykiDev can be used throught TCP/IP network  ... 

### The same, but through ZeroRPC

### First, a ZeroRPC server must be started.

From the command line:

$python ./smile.py /dev/ttyACM0 9600


If everything is correct, the command-line output should be:

n=None, name=None, com_port='/dev/ttyACM0', baudrate=  115200, bind_port= 2020 


0.   None   /dev/ttyS0


1.   Arduino (www.arduino.cc)   /dev/ttyACM0


Opening: /dev/ttyACM0 @ baudrate: 115200


Testing ... 


Arduino responsed on hello with: ['Arduino', 'SMILE']


Set mode to: 1


The zerorpc server's starting on: 


192.168.1.106:2020    <- insted this the ip address of your computer... :bind port  


The zerorpc server's runing!

### Starting the ZeroRPC client

In [1]:
import zerorpc
ard = zerorpc.Client()
ard.connect("tcp://127.0.0.1:2020")

[<SocketContext(connect='tcp://127.0.0.1:2020')>]

###  The rest is the same as before...

In [2]:
from pprint import pprint

print(f'\n\n  Returns the name of the project: {ard.hello() = }') #compare with above ...
print(f'\n\n  Sets mode of operation of the Arduino. Returns setted mode:  { ard.mode(3) = }') 
print('\n\n Variables:')
print('[name, value, min, max, update flag]')
pprint(ard.varlst())#
# ... or
print("\n\n List of all available Arduino command names:")
pprint(ard.hlp())





  Returns the name of the project: ard.hello() = ['Arduino', 'SMILE']


  Sets mode of operation of the Arduino. Returns setted mode:   ard.mode(3) = 3


 Variables:
[name, value, min, max, update flag]
[['t', 544.0, 0.0, 0.0, 1],
 ['f2', 5.72, 0.0, 0.0, 1],
 ['f1', 5.57, 0.0, 0.0, 1],
 ['a2', 2.0, 0.0, 10.0, 1],
 ['a1', 2.0, 0.0, 10.0, 1],
 ['w2', 0.1, 0.01, 10000.0, 1],
 ['w1', 1.0, 0.01, 10000.0, 1]]


 List of all available Arduino command names:
['varlst',
 't',
 'f2',
 'f1',
 'a2',
 'a1',
 'w2',
 'w1',
 'mode',
 'hello',
 'echo',
 'hlp']


In [3]:
ard.t()

544.0

In [4]:
ard.a1(2)
ard.a2(2)
ard.w1(1) # -direct  7Hz 20 points , zerorpc 4hZ
#ard.w2(17) 
#ard.w1(2)
#ard.w2(4)
y = []
z = []
y = [ard.f1() for i in range(40) ]
#z = [ard.f2() for i in range(100) ]
#%matplotlib notebook
%matplotlib qt
import matplotlib.pyplot as plt
#%matplotlib inline
plt.plot(y,'r*-',  label=f'f1 = {ard.w1()} Hz')
#plt.plot(z,'b*-',  label=f'f2 = {ard.w2()} Hz')

plt.title("Frequence Upper Limit Experiment")
plt.xlabel("Sample")
plt.ylabel("Signal")
plt.legend()


plt.show()



In [10]:
ard.w1(0.5) # - direct 0.5 Hz   
ard.w2(1)
ard.a1(3)
ard.a2(3)
%matplotlib qt

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
#%matplotlib inline
# --- init animation ---
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()  # втора ордината (споделен X)

xdata = []
y1data, y2data = [], []

line1, = ax1.plot([], [], 'r-', label='f1')  # червен сигнал
#line2, = ax2.plot([], [], 'b-', label='f2')  # син сигнал

ax1.set_xlabel("Time / Frames")
ax1.set_ylabel("sin(t) f1", color='r')
#ax2.set_ylabel("sin(t) f2", color='b')

max_points = 40

# ---  ---
def animate(i):
    v1 = ard.command('f1')
    #v2 = ard.command('f2')
    xdata.append(i)
    y1data.append(v1)
    #y2data.append(v2)
    plt.show()
    # ограничаваме броя на точките
    if len(xdata) > max_points:
        xdata.pop(0)
        y1data.pop(0)
        #y2data.pop(0)

    # обновяване на данните
    line1.set_data(range(len(y1data)), y1data)
    #line2.set_data(range(len(y2data)), y2data)

    # --- динамично мащабиране ---
    ax1.relim()          # преизчислява границите на оста
    ax1.autoscale_view() # мащабира според новите данни
    ax2.relim()
    ax2.autoscale_view()

    ax1.set_xlim(0, max_points)
    return line1 #, line2

anim = FuncAnimation(fig, animate, interval=1, blit=False, cache_frame_data=False)
plt.show()


In [28]:
ard.raw_lines('hlp')

[b'hlp\r\n',
 b'varlst\r\n',
 b't\r\n',
 b'f2\r\n',
 b'f1\r\n',
 b'a2\r\n',
 b'a1\r\n',
 b'w2\r\n',
 b'w1\r\n',
 b'mode\r\n',
 b'hello\r\n',
 b'echo\r\n',
 b'hlp\r\n',
 b'\r\n',
 b'>>']

In [12]:
ard.hlp() # Help – shows all available commands

['varlst',
 't',
 'f2',
 'f1',
 'a2',
 'a1',
 'w2',
 'w1',
 'mode',
 'hello',
 'echo',
 'hlp']

In [13]:
ard.command("hlp")   #The same as above but not mapped like a python function.

['varlst',
 't',
 'f2',
 'f1',
 'a2',
 'a1',
 'w2',
 'w1',
 'mode',
 'hello',
 'echo',
 'hlp']

In [14]:
ard.varlst() # List of all global variables: name, value, min, max, updated flag — not used at the moment.

[['t', 552.0, 0.0, 0.0, 1],
 ['f2', 8.89, 0.0, 0.0, 1],
 ['f1', 3.61, 0.0, 0.0, 1],
 ['a2', 3.0, 0.0, 10.0, 1],
 ['a1', 3.0, 0.0, 10.0, 1],
 ['w2', 1.0, 0.01, 10000.0, 1],
 ['w1', 0.5, 0.01, 10000.0, 1]]

In [5]:
del ard 